# Comprehensions => Scope & the Walrus Operator

Comprehensions have their own scope, and a few rules behave differently from normal loops.

| Topic | Rule |
|---|---|
| Loop variable | Stays inside the comprehension |
| Walrus `:=` | Binds in the **enclosing** scope |
| Closures | Late binding: use `lambda i=i: i` |
| `yield` | Not allowed inside a comprehension |
| Python 3.12 | Comprehensions in functions are inlined (PEP 709) |

---

## Comprehension Scope

A comprehension runs in its **own scope**. The loop variable stays inside it and does not leak into the surrounding code.

```python
i = "outer"
[i * i for i in range(3)]
print(i)          # "outer"  (unchanged)
```

Only the **leftmost iterable** is evaluated in the enclosing scope. The rest runs in the comprehension's own scope.

## Late Binding in Closures

A `lambda` created inside a comprehension remembers the **variable**, not its value at that moment.

```python
funcs = [lambda: i for i in range(3)]
[f() for f in funcs]              # [2, 2, 2]

funcs = [lambda i=i: i for i in range(3)]
[f() for f in funcs]              # [0, 1, 2]  (default argument captures the value)
```

## The Walrus Operator `:=`

An assignment expression (`name := value`) can be used inside a comprehension. Unlike the loop variable, it **binds in the enclosing scope**.

```python
[y for x in data if (y := f(x)) > 10]
```

Here `f(x)` is computed **once** and reused both in the condition and in the result.

- Put the walrus expression in **parentheses** inside the `if`.
- After the comprehension, `y` still exists in the surrounding scope.

### Walrus Restrictions

These raise `SyntaxError`:

| Code | Why |
|---|---|
| `[i := 0 for i in range(3)]` | Cannot rebind the iteration variable |
| `[x for x in (y := range(3))]` | Not allowed in the iterable expression |

## `yield` Is Not Allowed

`yield` and `yield from` are prohibited inside a comprehension (Python 3.8 and later).

## Python 3.12: Inlined Comprehensions (PEP 709)

Since Python 3.12, list, dict and set comprehensions inside functions are **inlined** instead of creating a hidden function each time. This makes them faster, up to 2x in a microbenchmark.

- The iteration variable is still isolated. The scoping rules above did not change.
- Tracebacks no longer show a separate frame for the comprehension.

## Key Rules

- The loop variable does not leak. The walrus variable does.
- Do not create closures in a comprehension without capturing the value.
- Use `:=` sparingly. If it makes the line hard to read, use a loop.

## Source

https://docs.python.org/3/reference/expressions.html

https://peps.python.org/pep-0572/

https://peps.python.org/pep-0709/

In [ ]:
# The loop variable does not leak
i = "outer"
squares = [i * i for i in range(3)]
print(i)                          # still "outer"

# The walrus operator DOES bind in the surrounding scope
data = [1, 5, 10, 20]
big = [y for x in data if (y := x * 2) > 10]
print(big, y)                     # y is visible here: the last value assigned

# Compute once, use twice
def expensive(n):
    return n * n

results = [r for n in range(6) if (r := expensive(n)) > 5]
print(results)

# Late binding in closures
funcs = [lambda: i for i in range(3)]
print([f() for f in funcs])                 # [2, 2, 2]: every lambda sees the final i

funcs_fixed = [lambda i=i: i for i in range(3)]
print([f() for f in funcs_fixed])           # [0, 1, 2]: the default argument captures the value

# Walrus restrictions raise SyntaxError (checked with compile, so nothing breaks here)
cases = {
    "rebind the iteration variable": "[i := 0 for i in range(3)]",
    "use it in the iterable expression": "[x for x in (y := range(3))]",
}
for reason, source in cases.items():
    try:
        compile(source, "<demo>", "exec")
    except SyntaxError:
        print("SyntaxError when you", reason)